# Spatial Data Preparation and Travel Graph Construction

This notebook constructs the **whole-Iligan** layered travel graph for the thesis.

It follows the clarified sequence:
1. extract the whole Iligan walk and drive graphs
2. save `nodes_uncategorized.csv`
3. save `nodes_walk.csv`
4. save `nodes_ride.csv` where **ride = drivable node layer**
5. build bidirectional start-walk edges
6. build bidirectional end-walk edges
7. build bidirectional ride/drivable edges
8. build **wait**
9. build **alight**
10. build **direct**
11. build **transfer**
12. stitch the layered graph
13. generate **zoomable HTML previews** for verification


## 0. Optional installation cell

In [96]:
# Uncomment and run only if needed.
%pip install osmnx geopandas folium networkx shapely pandas numpy


Note: you may need to restart the kernel to use updated packages.


In [97]:

from __future__ import annotations

from pathlib import Path
from IPython.display import IFrame, display

import numpy as np
import pandas as pd
import networkx as nx
import osmnx as ox
from shapely.geometry import LineString


## 1. Project configuration

In [98]:

# -----------------------------
# Study-area settings
# -----------------------------
PLACE_QUERY_CANDIDATES = [
    "Iligan City, Philippines"
]
POINT_FALLBACK_QUERY = "Iligan City, Philippines"
POINT_FALLBACK_DIST_M = 30000

# -----------------------------
# Matching and graph settings
# -----------------------------
COORD_ROUNDING = 7
MAX_INTERLAYER_SNAP_DIST_M = 80.0
SIMPLIFY_GRAPHS = True
RETAIN_ALL_COMPONENTS = True

# -----------------------------
# Project folders
# -----------------------------
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PREVIEW_DIR = OUTPUT_DIR / "previews"

for folder in [DATA_DIR, OUTPUT_DIR, PREVIEW_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Output file paths
# -----------------------------
NODES_UNCATEGORIZED_CSV = OUTPUT_DIR / "nodes_uncategorized.csv"
NODES_WALK_CSV = OUTPUT_DIR / "nodes_walk.csv"
NODES_RIDE_CSV = OUTPUT_DIR / "nodes_ride.csv"

EDGES_START_WALK_CSV = OUTPUT_DIR / "edges_start_walk.csv"
EDGES_END_WALK_CSV = OUTPUT_DIR / "edges_end_walk.csv"
EDGES_RIDE_CSV = OUTPUT_DIR / "edges_ride.csv"
EDGES_WAIT_CSV = OUTPUT_DIR / "edges_wait.csv"
EDGES_ALIGHT_CSV = OUTPUT_DIR / "edges_alight.csv"
EDGES_DIRECT_CSV = OUTPUT_DIR / "edges_direct.csv"
EDGES_TRANSFER_CSV = OUTPUT_DIR / "edges_transfer.csv"

TRAVEL_GRAPH_NODES_CSV = OUTPUT_DIR / "travel_graph_nodes.csv"
TRAVEL_GRAPH_EDGES_CSV = OUTPUT_DIR / "iligan_travel_graph.csv"

PREVIEW_WALK_NODES_HTML = PREVIEW_DIR / "preview_walk_nodes.html"
PREVIEW_RIDE_NODES_HTML = PREVIEW_DIR / "preview_ride_nodes.html"
PREVIEW_RIDE_NETWORK_HTML = PREVIEW_DIR / "preview_ride_network.html"
PREVIEW_STITCHED_HTML = PREVIEW_DIR / "preview_travel_graph_stitched.html"

ox.settings.use_cache = True
ox.settings.log_console = True


## 2. Helper functions

In [99]:

def make_coord_key(df: pd.DataFrame, lon_col: str = "lon", lat_col: str = "lat", decimals: int = 7) -> pd.Series:
    return df[lon_col].round(decimals).astype(str) + "|" + df[lat_col].round(decimals).astype(str)


def resolve_study_area_boundary(place_queries: list):
    last_error = None
    for query in place_queries:
        try:
            gdf = ox.geocode_to_gdf(query)
            if gdf.empty:
                continue

            chosen = gdf.copy()
            if {"class", "type"}.issubset(chosen.columns):
                mask = (
                    chosen["class"].astype(str).str.lower().eq("boundary")
                    & chosen["type"].astype(str).str.lower().eq("administrative")
                )
                if mask.any():
                    chosen = chosen.loc[mask].copy()

            chosen = chosen.iloc[[0]].copy()
            geom = chosen.geometry.iloc[0]
            if geom is None or geom.is_empty:
                continue

            return chosen, f"geocode_to_gdf({query!r})"
        except Exception as exc:
            last_error = exc

    if last_error is not None:
        raise RuntimeError(f"Unable to resolve the Iligan boundary: {last_error}")
    raise RuntimeError("Unable to resolve the Iligan boundary from the configured place queries.")


def load_graphs_for_study_area(
    place_queries: list,
    point_query: str | None = None,
    point_dist: float = 30000,
    simplify: bool = True,
    retain_all: bool = True,
):
    study_area_gdf, boundary_source = resolve_study_area_boundary(place_queries)
    polygon = study_area_gdf.geometry.union_all() if hasattr(study_area_gdf.geometry, "union_all") else study_area_gdf.unary_union

    try:
        G_walk_raw = ox.graph_from_polygon(
            polygon,
            network_type="walk",
            simplify=simplify,
            retain_all=retain_all,
        )
        G_drive_raw = ox.graph_from_polygon(
            polygon,
            network_type="drive",
            simplify=simplify,
            retain_all=retain_all,
        )
        graph_source = "graph_from_polygon(study_area_boundary)"
    except Exception as exc:
        if point_query is None:
            raise
        point = ox.geocode(point_query)
        G_walk_raw = ox.graph_from_point(
            point,
            dist=point_dist,
            network_type="walk",
            simplify=simplify,
            retain_all=retain_all,
        )
        G_drive_raw = ox.graph_from_point(
            point,
            dist=point_dist,
            network_type="drive",
            simplify=simplify,
            retain_all=retain_all,
        )
        graph_source = f"graph_from_point({point_query!r}, dist={point_dist}) because polygon download failed: {type(exc).__name__}: {exc}"

    G_walk_proj = ox.project_graph(G_walk_raw)
    G_drive_proj = ox.project_graph(G_drive_raw)
    return study_area_gdf, boundary_source, graph_source, G_walk_raw, G_walk_proj, G_drive_raw, G_drive_proj


def node_table_from_graph(G_raw: nx.MultiDiGraph, G_proj: nx.MultiDiGraph) -> pd.DataFrame:
    raw_nodes = pd.DataFrame.from_dict(dict(G_raw.nodes(data=True)), orient="index").reset_index()
    raw_nodes = raw_nodes.rename(columns={"index": "base_node_id", "x": "lon", "y": "lat"})

    proj_nodes = pd.DataFrame.from_dict(dict(G_proj.nodes(data=True)), orient="index").reset_index()
    proj_nodes = proj_nodes.rename(columns={"index": "base_node_id", "x": "x", "y": "y"})

    merged = proj_nodes[["base_node_id", "x", "y"]].merge(
        raw_nodes[["base_node_id", "lon", "lat"]],
        on="base_node_id",
        how="left",
    )
    merged["coord_key"] = make_coord_key(merged, "lon", "lat", COORD_ROUNDING)
    merged["node_id"] = merged["base_node_id"].astype(str)
    return merged[["node_id", "base_node_id", "x", "y", "lon", "lat", "coord_key"]].sort_values("base_node_id").reset_index(drop=True)


def extract_uncategorized_nodes(walk_nodes: pd.DataFrame, drive_nodes: pd.DataFrame) -> pd.DataFrame:
    walk = walk_nodes.copy()
    walk["in_walk_graph"] = True
    walk["in_drive_graph"] = False

    drive = drive_nodes.copy()
    drive["in_walk_graph"] = False
    drive["in_drive_graph"] = True

    union = pd.concat([walk, drive], ignore_index=True, sort=False)
    union = union.groupby("base_node_id", as_index=False).agg(
        {
            "node_id": "first",
            "x": "first",
            "y": "first",
            "lon": "first",
            "lat": "first",
            "coord_key": "first",
            "in_walk_graph": "max",
            "in_drive_graph": "max",
        }
    )
    union["node_id"] = union["base_node_id"].astype(str)
    return union[
        ["node_id", "base_node_id", "x", "y", "lon", "lat", "in_walk_graph", "in_drive_graph", "coord_key"]
    ].sort_values("base_node_id").reset_index(drop=True)


def graph_edges_to_bidirectional_base(G_proj: nx.MultiDiGraph, prefix: str, edge_type: str) -> pd.DataFrame:
    edges_gdf = ox.graph_to_gdfs(G_proj, nodes=False, edges=True).reset_index()

    if "length" not in edges_gdf.columns:
        edges_gdf["length"] = edges_gdf.geometry.length

    edges_gdf["pair_key"] = edges_gdf.apply(lambda row: tuple(sorted((row["u"], row["v"]))), axis=1)
    edges_gdf = (
        edges_gdf.sort_values(["pair_key", "length"])
        .drop_duplicates(subset=["pair_key"], keep="first")
        .copy()
    )

    rows = []
    for row in edges_gdf.itertuples(index=False):
        pair_u, pair_v = row.pair_key
        dist = float(row.length)
        geom = row.geometry

        rows.append(
            {
                "u": f"{prefix}_{pair_u}",
                "v": f"{prefix}_{pair_v}",
                "dist": dist,
                "edge_type": edge_type,
                "geometry": geom,
            }
        )
        rev_geom = LineString(list(geom.coords)[::-1]) if geom is not None else None
        rows.append(
            {
                "u": f"{prefix}_{pair_v}",
                "v": f"{prefix}_{pair_u}",
                "dist": dist,
                "edge_type": edge_type,
                "geometry": rev_geom,
            }
        )
    return pd.DataFrame(rows)


def duplicate_walk_nodes_to_layers(nodes_walk: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    start_nodes = nodes_walk.copy()
    start_nodes["node_id"] = "sw_" + start_nodes["base_node_id"].astype(str)

    end_nodes = nodes_walk.copy()
    end_nodes["node_id"] = "ew_" + end_nodes["base_node_id"].astype(str)

    return start_nodes, end_nodes


def build_direct_edges(start_nodes: pd.DataFrame, end_nodes: pd.DataFrame) -> pd.DataFrame:
    merged = start_nodes[["base_node_id", "node_id", "x", "y"]].merge(
        end_nodes[["base_node_id", "node_id", "x", "y"]],
        on="base_node_id",
        how="inner",
        suffixes=("_start", "_end"),
    )

    rows = []
    for row in merged.itertuples(index=False):
        rows.append(
            {
                "u": row.node_id_start,
                "v": row.node_id_end,
                "dist": 0.0,
                "edge_type": "direct",
                "geometry": LineString([(row.x_start, row.y_start), (row.x_end, row.y_end)]),
            }
        )
    return pd.DataFrame(rows)


def _prepare_match_df(df: pd.DataFrame, node_col: str) -> pd.DataFrame:
    return df[[node_col, "base_node_id", "x", "y", "lon", "lat", "coord_key"]].drop_duplicates(node_col).reset_index(drop=True)


def build_interlayer_edges(
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    left_node_col: str,
    right_node_col: str,
    edge_type: str,
    max_snap_dist_m: float | None = None,
    anchor_side: str = "right",
) -> pd.DataFrame:
    if left_df.empty or right_df.empty:
        return pd.DataFrame(columns=["u", "v", "dist", "edge_type", "geometry"])

    if anchor_side not in {"left", "right"}:
        raise ValueError("anchor_side must be 'left' or 'right'.")

    source_df = left_df if anchor_side == "left" else right_df
    target_df = right_df if anchor_side == "left" else left_df
    source_node_col = left_node_col if anchor_side == "left" else right_node_col
    target_node_col = right_node_col if anchor_side == "left" else left_node_col

    source = _prepare_match_df(source_df, source_node_col)
    target = _prepare_match_df(target_df, target_node_col)

    records = []

    exact = source.merge(target, on="coord_key", how="inner", suffixes=("_source", "_target"))
    source_id_key = source_node_col + "_source" if source_node_col + "_source" in exact.columns else source_node_col
    target_id_key = target_node_col + "_target" if target_node_col + "_target" in exact.columns else target_node_col

    exact_source_ids = set()
    for rec in exact.to_dict("records"):
        exact_source_ids.add(rec[source_id_key])
        records.append(
            {
                "source_node": rec[source_id_key],
                "target_node": rec[target_id_key],
                "x_source": rec["x_source"],
                "y_source": rec["y_source"],
                "x_target": rec["x_target"],
                "y_target": rec["y_target"],
            }
        )

    source_remaining = source[~source[source_node_col].isin(exact_source_ids)].copy()
    if not source_remaining.empty and not target.empty:
        target_xy = target[["x", "y"]].to_numpy(dtype=float)
        for rec in source_remaining.to_dict("records"):
            sx = float(rec["x"])
            sy = float(rec["y"])
            dists = np.sqrt((target_xy[:, 0] - sx) ** 2 + (target_xy[:, 1] - sy) ** 2)
            nearest_idx = int(dists.argmin())
            snap_dist = float(dists[nearest_idx])

            if max_snap_dist_m is not None and snap_dist > float(max_snap_dist_m):
                continue

            target_row = target.iloc[nearest_idx].to_dict()
            records.append(
                {
                    "source_node": rec[source_node_col],
                    "target_node": target_row[target_node_col],
                    "x_source": rec["x"],
                    "y_source": rec["y"],
                    "x_target": target_row["x"],
                    "y_target": target_row["y"],
                }
            )

    rows = []
    for rec in records:
        if anchor_side == "left":
            u = rec["source_node"]
            v = rec["target_node"]
            x_u, y_u = rec["x_source"], rec["y_source"]
            x_v, y_v = rec["x_target"], rec["y_target"]
        else:
            u = rec["target_node"]
            v = rec["source_node"]
            x_u, y_u = rec["x_target"], rec["y_target"]
            x_v, y_v = rec["x_source"], rec["y_source"]

        rows.append(
            {
                "u": u,
                "v": v,
                "dist": 0.0,
                "edge_type": edge_type,
                "geometry": LineString([(x_u, y_u), (x_v, y_v)]),
            }
        )
    return pd.DataFrame(rows)


def assign_edge_ids(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    out = df.copy().reset_index(drop=True)
    out.insert(0, "edge_id", [f"{prefix}_{i + 1:06d}" for i in range(len(out))])
    return out


def edge_frame_to_csv(df: pd.DataFrame, csv_path: Path) -> None:
    out = df.copy()
    save_cols = [col for col in ["edge_id", "u", "v", "dist", "edge_type"] if col in out.columns]
    out = out[save_cols]
    out.to_csv(csv_path, index=False)


def node_frame_to_csv(df: pd.DataFrame, csv_path: Path, keep_cols: list[str]) -> None:
    out = df.copy()
    out = out[keep_cols]
    out.to_csv(csv_path, index=False)


def make_edges_gdf(df: pd.DataFrame, crs):
    import geopandas as gpd
    out = df.copy()
    if "geometry" not in out.columns:
        out["geometry"] = pd.Series(dtype="object")
    return gpd.GeoDataFrame(out, geometry="geometry", crs=crs)


def make_nodes_gdf(df: pd.DataFrame, crs):
    import geopandas as gpd
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df["x"], df["y"]),
        crs=crs,
    )
    return gdf


def add_nodes_to_digraph(G: nx.DiGraph, nodes_df: pd.DataFrame) -> None:
    for row in nodes_df.itertuples(index=False):
        attrs = row._asdict()
        node_id = attrs.pop("node_id")
        attrs.pop("coord_key", None)
        G.add_node(node_id, **attrs)


def add_edges_to_digraph(G: nx.DiGraph, edges_df: pd.DataFrame) -> None:
    for row in edges_df.itertuples(index=False):
        attrs = row._asdict()
        edge_id = attrs.pop("edge_id")
        geom = attrs.pop("geometry", None)
        u = attrs.pop("u")
        v = attrs.pop("v")
        G.add_edge(u, v, edge_id=edge_id, geometry=geom, **attrs)


def add_gdf_to_folium(
    fmap,
    gdf,
    name: str,
    color: str = "#3388ff",
    weight: float = 2,
    opacity: float = 0.8,
    fill_opacity: float = 0.1,
    show: bool = True,
    popup_fields: list[str] | None = None,
    is_point_layer: bool = False,
):
    import folium

    if gdf is None or len(gdf) == 0:
        return

    gdf_ll = gdf.to_crs(4326) if str(gdf.crs) != "EPSG:4326" else gdf

    popup = None
    if popup_fields:
        valid_fields = [f for f in popup_fields if f in gdf_ll.columns]
        if valid_fields:
            popup = folium.GeoJsonPopup(fields=valid_fields, labels=True)

    style_kwargs = {
        "color": color,
        "weight": weight,
        "opacity": opacity,
        "fillOpacity": fill_opacity,
    }
    if is_point_layer:
        style_kwargs["radius"] = 2

    folium.GeoJson(
        gdf_ll,
        name=name,
        show=show,
        style_function=lambda _x, style_kwargs=style_kwargs: style_kwargs,
        marker=folium.CircleMarker(radius=2, color=color, fill=True, fill_opacity=0.8) if is_point_layer else None,
        popup=popup,
        zoom_on_click=False,
    ).add_to(fmap)


def create_interactive_map(
    boundary_gdf,
    output_html_path: Path,
    title: str,
    walk_nodes_gdf=None,
    ride_nodes_gdf=None,
    walk_edges_gdf=None,
    ride_edges_gdf=None,
    wait_edges_gdf=None,
    alight_edges_gdf=None,
    direct_edges_gdf=None,
    transfer_edges_gdf=None,
):
    import folium
    from folium.plugins import Fullscreen

    boundary_ll = boundary_gdf.to_crs(4326) if str(boundary_gdf.crs) != "EPSG:4326" else boundary_gdf
    centroid = boundary_ll.geometry.iloc[0].centroid

    fmap = folium.Map(
        location=[centroid.y, centroid.x],
        zoom_start=12,
        tiles="CartoDB positron",
        control_scale=True,
    )
    Fullscreen().add_to(fmap)

    title_html = f'''
         <div style="position: fixed;
                     top: 10px; left: 50px; width: 440px; z-index:9999;
                     background-color: white; padding: 10px; border: 1px solid #999;">
             <b>{title}</b>
         </div>
    '''
    fmap.get_root().html.add_child(folium.Element(title_html))

    add_gdf_to_folium(
        fmap,
        boundary_ll,
        name="Iligan boundary",
        color="#111111",
        weight=2.5,
        opacity=1.0,
        fill_opacity=0.02,
        show=True,
    )

    add_gdf_to_folium(
        fmap,
        walk_nodes_gdf,
        name="Walk nodes",
        color="#1f77b4",
        weight=1,
        opacity=0.9,
        fill_opacity=0.8,
        show=True,
        popup_fields=["node_id", "base_node_id", "lon", "lat"],
        is_point_layer=True,
    )
    add_gdf_to_folium(
        fmap,
        ride_nodes_gdf,
        name="Drivable nodes",
        color="#d62728",
        weight=1,
        opacity=0.9,
        fill_opacity=0.8,
        show=True,
        popup_fields=["node_id", "base_node_id", "lon", "lat"],
        is_point_layer=True,
    )
    add_gdf_to_folium(
        fmap,
        walk_edges_gdf,
        name="Walk edges",
        color="#7f7f7f",
        weight=1.2,
        opacity=0.55,
        show=False,
        popup_fields=["edge_id", "u", "v", "dist", "edge_type"],
    )
    add_gdf_to_folium(
        fmap,
        ride_edges_gdf,
        name="Drivable edges",
        color="#d62728",
        weight=2.2,
        opacity=0.9,
        show=True,
        popup_fields=["edge_id", "u", "v", "dist", "edge_type"],
    )
    add_gdf_to_folium(
        fmap,
        wait_edges_gdf,
        name="Wait edges",
        color="#2ca02c",
        weight=1.8,
        opacity=0.8,
        show=False,
        popup_fields=["edge_id", "u", "v", "edge_type"],
    )
    add_gdf_to_folium(
        fmap,
        alight_edges_gdf,
        name="Alight edges",
        color="#9467bd",
        weight=1.8,
        opacity=0.8,
        show=False,
        popup_fields=["edge_id", "u", "v", "edge_type"],
    )
    add_gdf_to_folium(
        fmap,
        direct_edges_gdf,
        name="Direct edges",
        color="#8c564b",
        weight=1.6,
        opacity=0.8,
        show=False,
        popup_fields=["edge_id", "u", "v", "edge_type"],
    )
    add_gdf_to_folium(
        fmap,
        transfer_edges_gdf,
        name="Transfer edges",
        color="#ff7f0e",
        weight=1.8,
        opacity=0.8,
        show=False,
        popup_fields=["edge_id", "u", "v", "edge_type"],
    )

    bounds = boundary_ll.total_bounds
    fmap.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    folium.LayerControl(collapsed=False).add_to(fmap)
    fmap.save(str(output_html_path))
    return fmap


def show_html_preview(path: Path, height: int = 720):
    if path.exists():
        display(IFrame(src=str(path), width="100%", height=height))
    else:
        print(f"Preview file does not exist yet: {path}")


## 3. Extract the whole Iligan walk and drive graphs

In [100]:

study_area_gdf, boundary_source, graph_source, G_walk_raw, G_walk_proj, G_drive_raw, G_drive_proj = load_graphs_for_study_area(
    place_queries=PLACE_QUERY_CANDIDATES,
    point_query=POINT_FALLBACK_QUERY,
    point_dist=POINT_FALLBACK_DIST_M,
    simplify=SIMPLIFY_GRAPHS,
    retain_all=RETAIN_ALL_COMPONENTS,
)

print("Boundary source:", boundary_source)
print("Graph source:", graph_source)
print("Walk graph:", len(G_walk_proj.nodes), "nodes |", len(G_walk_proj.edges), "edges")
print("Drive graph:", len(G_drive_proj.nodes), "nodes |", len(G_drive_proj.edges), "edges")
print("Projected CRS:", G_walk_proj.graph.get("crs"))


Boundary source: geocode_to_gdf('Iligan City, Philippines')
Graph source: graph_from_polygon(study_area_boundary)
Walk graph: 4689 nodes | 12014 edges
Drive graph: 3477 nodes | 8851 edges
Projected CRS: EPSG:32651


## 4. Extract raw nodes plus attributes and save `nodes_uncategorized.csv`

In [101]:

walk_nodes_lookup = node_table_from_graph(G_walk_raw, G_walk_proj).copy()
drive_nodes_lookup = node_table_from_graph(G_drive_raw, G_drive_proj).copy()

nodes_uncategorized = extract_uncategorized_nodes(walk_nodes_lookup, drive_nodes_lookup)
node_frame_to_csv(
    nodes_uncategorized,
    NODES_UNCATEGORIZED_CSV,
    keep_cols=["node_id", "base_node_id", "x", "y", "lon", "lat", "in_walk_graph", "in_drive_graph"],
)

print("Saved:", NODES_UNCATEGORIZED_CSV)
nodes_uncategorized.head()


Saved: /Users/apple/Documents/vscode/Projects/Thesis/notebooks/outputs/nodes_uncategorized.csv


,node_id,base_node_id,x,y,lon,lat,in_walk_graph,in_drive_graph,coord_key
0,245366634,245366634,638516.970881,904758.848210,124.257439,8.183123,True,True,124.2574391|8.183123
1,245366694,245366694,637154.936706,907971.088369,124.245167,8.212211,True,True,124.2451671|8.2122106
2,333797343,333797343,637021.741374,909309.974211,124.243996,8.224322,True,True,124.2439958|8.2243223
3,333797496,333797496,637053.410934,909364.013039,124.244285,8.224810,True,True,124.2442848|8.2248101
4,333797842,333797842,637344.086758,910507.089225,124.246956,8.235139,True,False,124.2469557|8.2351391


## 5. Treat all walk-graph nodes as walk nodes and save `nodes_walk.csv`

In [102]:

nodes_walk = walk_nodes_lookup.copy()[["node_id", "base_node_id", "x", "y", "lon", "lat", "coord_key"]]
node_frame_to_csv(
    nodes_walk,
    NODES_WALK_CSV,
    keep_cols=["node_id", "base_node_id", "x", "y", "lon", "lat"],
)

print("Saved:", NODES_WALK_CSV)
nodes_walk.head()


Saved: /Users/apple/Documents/vscode/Projects/Thesis/notebooks/outputs/nodes_walk.csv


,node_id,base_node_id,x,y,lon,lat,coord_key
0,245366634,245366634,638516.970881,904758.848210,124.257439,8.183123,124.2574391|8.183123
1,245366694,245366694,637154.936706,907971.088369,124.245167,8.212211,124.2451671|8.2122106
2,333797343,333797343,637021.741374,909309.974211,124.243996,8.224322,124.2439958|8.2243223
3,333797496,333797496,637053.410934,909364.013039,124.244285,8.224810,124.2442848|8.2248101
4,333797842,333797842,637344.086758,910507.089225,124.246956,8.235139,124.2469557|8.2351391


## 6. Treat all drive-graph nodes as drivable ride nodes and save `nodes_ride.csv`

In [103]:

nodes_ride = drive_nodes_lookup.copy()[["node_id", "base_node_id", "x", "y", "lon", "lat", "coord_key"]]
node_frame_to_csv(
    nodes_ride,
    NODES_RIDE_CSV,
    keep_cols=["node_id", "base_node_id", "x", "y", "lon", "lat"],
)

print("Saved:", NODES_RIDE_CSV)
nodes_ride.head()


Saved: /Users/apple/Documents/vscode/Projects/Thesis/notebooks/outputs/nodes_ride.csv


,node_id,base_node_id,x,y,lon,lat,coord_key
0,245366634,245366634,638516.970881,904758.848210,124.257439,8.183123,124.2574391|8.183123
1,245366694,245366694,637154.936706,907971.088369,124.245167,8.212211,124.2451671|8.2122106
2,333797343,333797343,637021.741374,909309.974211,124.243996,8.224322,124.2439958|8.2243223
3,333797496,333797496,637053.410934,909364.013039,124.244285,8.224810,124.2442848|8.2248101
4,333801115,333801115,628899.600869,905210.052231,124.170159,8.187466,124.1701587|8.1874658


## 7. Preview the whole-Iligan walk nodes

In [104]:

walk_nodes_preview_gdf = make_nodes_gdf(nodes_walk, crs=G_walk_proj.graph["crs"])
create_interactive_map(
    boundary_gdf=study_area_gdf,
    output_html_path=PREVIEW_WALK_NODES_HTML,
    title="Whole Iligan - Walk Nodes Preview",
    walk_nodes_gdf=walk_nodes_preview_gdf,
)
show_html_preview(PREVIEW_WALK_NODES_HTML)


## 8. Preview the whole-Iligan drivable nodes

In [105]:

ride_nodes_preview_gdf = make_nodes_gdf(nodes_ride, crs=G_drive_proj.graph["crs"])
create_interactive_map(
    boundary_gdf=study_area_gdf,
    output_html_path=PREVIEW_RIDE_NODES_HTML,
    title="Whole Iligan - Drivable Nodes Preview",
    ride_nodes_gdf=ride_nodes_preview_gdf,
)
show_html_preview(PREVIEW_RIDE_NODES_HTML)


## 9. Build the bidirectional start-walk edges

In [106]:

edges_start_walk = graph_edges_to_bidirectional_base(
    G_proj=G_walk_proj,
    prefix="sw",
    edge_type="start_walk",
)
edges_start_walk = assign_edge_ids(edges_start_walk, "sw_edge")
edge_frame_to_csv(edges_start_walk, EDGES_START_WALK_CSV)

print("Saved:", EDGES_START_WALK_CSV)
edges_start_walk.head()


Saved: /Users/apple/Documents/vscode/Projects/Thesis/notebooks/outputs/edges_start_walk.csv


,edge_id,u,v,dist,edge_type,geometry
0,sw_edge_000001,sw_245366634,sw_8099897878,24.990711,start_walk,LINESTRING (638516.9708810493 904758.848210290...
1,sw_edge_000002,sw_8099897878,sw_245366634,24.990711,start_walk,LINESTRING (638509.3938900122 904782.532747543...
2,sw_edge_000003,sw_245366634,sw_12599690557,385.609143,start_walk,LINESTRING (638516.9708810493 904758.848210290...
3,sw_edge_000004,sw_12599690557,sw_245366634,385.609143,start_walk,LINESTRING (638479.2545532953 904900.902229016...
4,sw_edge_000005,sw_245366634,sw_12599690568,34.099817,start_walk,LINESTRING (638516.9708810493 904758.848210290...


## 10. Build the bidirectional end-walk edges

In [107]:

edges_end_walk = graph_edges_to_bidirectional_base(
    G_proj=G_walk_proj,
    prefix="ew",
    edge_type="end_walk",
)
edges_end_walk = assign_edge_ids(edges_end_walk, "ew_edge")
edge_frame_to_csv(edges_end_walk, EDGES_END_WALK_CSV)

print("Saved:", EDGES_END_WALK_CSV)
edges_end_walk.head()


Saved: /Users/apple/Documents/vscode/Projects/Thesis/notebooks/outputs/edges_end_walk.csv


,edge_id,u,v,dist,edge_type,geometry
0,ew_edge_000001,ew_245366634,ew_8099897878,24.990711,end_walk,LINESTRING (638516.9708810493 904758.848210290...
1,ew_edge_000002,ew_8099897878,ew_245366634,24.990711,end_walk,LINESTRING (638509.3938900122 904782.532747543...
2,ew_edge_000003,ew_245366634,ew_12599690557,385.609143,end_walk,LINESTRING (638516.9708810493 904758.848210290...
3,ew_edge_000004,ew_12599690557,ew_245366634,385.609143,end_walk,LINESTRING (638479.2545532953 904900.902229016...
4,ew_edge_000005,ew_245366634,ew_12599690568,34.099817,end_walk,LINESTRING (638516.9708810493 904758.848210290...


## 11. Build the bidirectional drivable ride edges

In [108]:

edges_ride = graph_edges_to_bidirectional_base(
    G_proj=G_drive_proj,
    prefix="ride",
    edge_type="ride",
)
edges_ride = assign_edge_ids(edges_ride, "ride_edge")
edge_frame_to_csv(edges_ride, EDGES_RIDE_CSV)

print("Saved:", EDGES_RIDE_CSV)
edges_ride.head()


Saved: /Users/apple/Documents/vscode/Projects/Thesis/notebooks/outputs/edges_ride.csv


,edge_id,u,v,dist,edge_type,geometry
0,ride_edge_000001,ride_245366634,ride_1030442070,119.989863,ride,LINESTRING (638528.6552566248 904640.653288030...
1,ride_edge_000002,ride_1030442070,ride_245366634,119.989863,ride,LINESTRING (638516.9708810493 904758.848210290...
2,ride_edge_000003,ride_245366634,ride_8099897878,24.990711,ride,LINESTRING (638516.9708810493 904758.848210290...
3,ride_edge_000004,ride_8099897878,ride_245366634,24.990711,ride,LINESTRING (638509.3938900122 904782.532747543...
4,ride_edge_000005,ride_245366634,ride_12599690567,190.082647,ride,LINESTRING (638516.9708810493 904758.848210290...


## 12. Preview the whole-Iligan drivable network

In [109]:

ride_edges_preview_gdf = make_edges_gdf(edges_ride.copy(), crs=G_drive_proj.graph["crs"])
create_interactive_map(
    boundary_gdf=study_area_gdf,
    output_html_path=PREVIEW_RIDE_NETWORK_HTML,
    title="Whole Iligan - Drivable Network Preview",
    ride_nodes_gdf=ride_nodes_preview_gdf,
    ride_edges_gdf=ride_edges_preview_gdf,
)
show_html_preview(PREVIEW_RIDE_NETWORK_HTML)


## 13. Duplicate the walk nodes into start-walk and end-walk layers

In [110]:

nodes_start_walk, nodes_end_walk = duplicate_walk_nodes_to_layers(nodes_walk)

print("Start-walk nodes:", len(nodes_start_walk))
print("End-walk nodes:", len(nodes_end_walk))
nodes_start_walk.head()


Start-walk nodes: 4689
End-walk nodes: 4689


,node_id,base_node_id,x,y,lon,lat,coord_key
0,sw_245366634,245366634,638516.970881,904758.848210,124.257439,8.183123,124.2574391|8.183123
1,sw_245366694,245366694,637154.936706,907971.088369,124.245167,8.212211,124.2451671|8.2122106
2,sw_333797343,333797343,637021.741374,909309.974211,124.243996,8.224322,124.2439958|8.2243223
3,sw_333797496,333797496,637053.410934,909364.013039,124.244285,8.224810,124.2442848|8.2248101
4,sw_333797842,333797842,637344.086758,910507.089225,124.246956,8.235139,124.2469557|8.2351391


## 14. Rename the drivable nodes into the ride-layer node IDs

In [111]:

nodes_ride_layer = nodes_ride.copy()
nodes_ride_layer["node_id"] = "ride_" + nodes_ride_layer["base_node_id"].astype(str)

print("Ride-layer nodes:", len(nodes_ride_layer))
nodes_ride_layer.head()


Ride-layer nodes: 3477


,node_id,base_node_id,x,y,lon,lat,coord_key
0,ride_245366634,245366634,638516.970881,904758.848210,124.257439,8.183123,124.2574391|8.183123
1,ride_245366694,245366694,637154.936706,907971.088369,124.245167,8.212211,124.2451671|8.2122106
2,ride_333797343,333797343,637021.741374,909309.974211,124.243996,8.224322,124.2439958|8.2243223
3,ride_333797496,333797496,637053.410934,909364.013039,124.244285,8.224810,124.2442848|8.2248101
4,ride_333801115,333801115,628899.600869,905210.052231,124.170159,8.187466,124.1701587|8.1874658


## 15. Build wait edges: `start_walk -> ride`

In [112]:

edges_wait = build_interlayer_edges(
    left_df=nodes_start_walk,
    right_df=nodes_ride_layer,
    left_node_col="node_id",
    right_node_col="node_id",
    edge_type="wait",
    max_snap_dist_m=MAX_INTERLAYER_SNAP_DIST_M,
    anchor_side="right",
)
edges_wait = assign_edge_ids(edges_wait, "wait_edge")
edge_frame_to_csv(edges_wait, EDGES_WAIT_CSV)

print("Saved:", EDGES_WAIT_CSV)
edges_wait.head()


Saved: /Users/apple/Documents/vscode/Projects/Thesis/notebooks/outputs/edges_wait.csv


,edge_id,u,v,dist,edge_type,geometry
0,wait_edge_000001,sw_245366634,ride_245366634,0.0,wait,LINESTRING (638516.9708810493 904758.848210290...
1,wait_edge_000002,sw_245366694,ride_245366694,0.0,wait,LINESTRING (637154.9367063687 907971.088368548...
2,wait_edge_000003,sw_333797343,ride_333797343,0.0,wait,LINESTRING (637021.7413736106 909309.974211277...
3,wait_edge_000004,sw_333797496,ride_333797496,0.0,wait,LINESTRING (637053.4109339686 909364.013038886...
4,wait_edge_000005,sw_333801115,ride_333801115,0.0,wait,LINESTRING (628899.6008685873 905210.052231287...


## 16. Build alight edges: `ride -> end_walk`

In [113]:

edges_alight = build_interlayer_edges(
    left_df=nodes_ride_layer,
    right_df=nodes_end_walk,
    left_node_col="node_id",
    right_node_col="node_id",
    edge_type="alight",
    max_snap_dist_m=MAX_INTERLAYER_SNAP_DIST_M,
    anchor_side="left",
)
edges_alight = assign_edge_ids(edges_alight, "alight_edge")
edge_frame_to_csv(edges_alight, EDGES_ALIGHT_CSV)

print("Saved:", EDGES_ALIGHT_CSV)
edges_alight.head()


Saved: /Users/apple/Documents/vscode/Projects/Thesis/notebooks/outputs/edges_alight.csv


,edge_id,u,v,dist,edge_type,geometry
0,alight_edge_000001,ride_245366634,ew_245366634,0.0,alight,LINESTRING (638516.9708810493 904758.848210290...
1,alight_edge_000002,ride_245366694,ew_245366694,0.0,alight,LINESTRING (637154.9367063687 907971.088368548...
2,alight_edge_000003,ride_333797343,ew_333797343,0.0,alight,LINESTRING (637021.7413736106 909309.974211277...
3,alight_edge_000004,ride_333797496,ew_333797496,0.0,alight,LINESTRING (637053.4109339686 909364.013038886...
4,alight_edge_000005,ride_333801115,ew_333801115,0.0,alight,LINESTRING (628899.6008685873 905210.052231287...


## 17. Build direct edges: `start_walk -> end_walk`

In [114]:

edges_direct = build_direct_edges(nodes_start_walk, nodes_end_walk)
edges_direct = assign_edge_ids(edges_direct, "direct_edge")
edge_frame_to_csv(edges_direct, EDGES_DIRECT_CSV)

print("Saved:", EDGES_DIRECT_CSV)
edges_direct.head()


Saved: /Users/apple/Documents/vscode/Projects/Thesis/notebooks/outputs/edges_direct.csv


,edge_id,u,v,dist,edge_type,geometry
0,direct_edge_000001,sw_245366634,ew_245366634,0.0,direct,LINESTRING (638516.9708810493 904758.848210290...
1,direct_edge_000002,sw_245366694,ew_245366694,0.0,direct,LINESTRING (637154.9367063687 907971.088368548...
2,direct_edge_000003,sw_333797343,ew_333797343,0.0,direct,LINESTRING (637021.7413736106 909309.974211277...
3,direct_edge_000004,sw_333797496,ew_333797496,0.0,direct,LINESTRING (637053.4109339686 909364.013038886...
4,direct_edge_000005,sw_333797842,ew_333797842,0.0,direct,LINESTRING (637344.0867584045 910507.089225195...


## 18. Build transfer edges: `end_walk -> ride`

In [115]:

edges_transfer = build_interlayer_edges(
    left_df=nodes_end_walk,
    right_df=nodes_ride_layer,
    left_node_col="node_id",
    right_node_col="node_id",
    edge_type="transfer",
    max_snap_dist_m=MAX_INTERLAYER_SNAP_DIST_M,
    anchor_side="right",
)
edges_transfer = assign_edge_ids(edges_transfer, "transfer_edge")
edge_frame_to_csv(edges_transfer, EDGES_TRANSFER_CSV)

print("Saved:", EDGES_TRANSFER_CSV)
edges_transfer.head()


Saved: /Users/apple/Documents/vscode/Projects/Thesis/notebooks/outputs/edges_transfer.csv


,edge_id,u,v,dist,edge_type,geometry
0,transfer_edge_000001,ew_245366634,ride_245366634,0.0,transfer,LINESTRING (638516.9708810493 904758.848210290...
1,transfer_edge_000002,ew_245366694,ride_245366694,0.0,transfer,LINESTRING (637154.9367063687 907971.088368548...
2,transfer_edge_000003,ew_333797343,ride_333797343,0.0,transfer,LINESTRING (637021.7413736106 909309.974211277...
3,transfer_edge_000004,ew_333797496,ride_333797496,0.0,transfer,LINESTRING (637053.4109339686 909364.013038886...
4,transfer_edge_000005,ew_333801115,ride_333801115,0.0,transfer,LINESTRING (628899.6008685873 905210.052231287...


## 19. Stitch the graph together

In [116]:

travel_graph_nodes = pd.concat(
    [
        nodes_start_walk.assign(layer="start_walk"),
        nodes_ride_layer.assign(layer="ride"),
        nodes_end_walk.assign(layer="end_walk"),
    ],
    ignore_index=True,
    sort=False,
)[["node_id", "base_node_id", "x", "y", "lon", "lat", "layer"]].drop_duplicates("node_id").reset_index(drop=True)

travel_graph_edges = pd.concat(
    [
        edges_start_walk,
        edges_end_walk,
        edges_ride,
        edges_wait,
        edges_alight,
        edges_direct,
        edges_transfer,
    ],
    ignore_index=True,
    sort=False,
)

travel_graph_nodes.to_csv(TRAVEL_GRAPH_NODES_CSV, index=False)
edge_frame_to_csv(travel_graph_edges, TRAVEL_GRAPH_EDGES_CSV)

print("Saved:", TRAVEL_GRAPH_NODES_CSV)
print("Saved:", TRAVEL_GRAPH_EDGES_CSV)
print("Total layered nodes:", len(travel_graph_nodes))
print("Total layered edges:", len(travel_graph_edges))
travel_graph_edges.head()


Saved: /Users/apple/Documents/vscode/Projects/Thesis/notebooks/outputs/travel_graph_nodes.csv
Saved: /Users/apple/Documents/vscode/Projects/Thesis/notebooks/outputs/iligan_travel_graph.csv
Total layered nodes: 12855
Total layered edges: 47507


,edge_id,u,v,dist,edge_type,geometry
0,sw_edge_000001,sw_245366634,sw_8099897878,24.990711,start_walk,LINESTRING (638516.9708810493 904758.848210290...
1,sw_edge_000002,sw_8099897878,sw_245366634,24.990711,start_walk,LINESTRING (638509.3938900122 904782.532747543...
2,sw_edge_000003,sw_245366634,sw_12599690557,385.609143,start_walk,LINESTRING (638516.9708810493 904758.848210290...
3,sw_edge_000004,sw_12599690557,sw_245366634,385.609143,start_walk,LINESTRING (638479.2545532953 904900.902229016...
4,sw_edge_000005,sw_245366634,sw_12599690568,34.099817,start_walk,LINESTRING (638516.9708810493 904758.848210290...


## 20. Build the in-memory directed graph object

In [117]:

travel_graph_nx = nx.DiGraph()
add_nodes_to_digraph(travel_graph_nx, travel_graph_nodes)
add_edges_to_digraph(travel_graph_nx, travel_graph_edges)

print("Directed graph nodes:", travel_graph_nx.number_of_nodes())
print("Directed graph edges:", travel_graph_nx.number_of_edges())


Directed graph nodes: 12855
Directed graph edges: 47463


## 21. Interactive stitched preview for verification

In [118]:

walk_edges_preview_gdf = make_edges_gdf(
    pd.concat([edges_start_walk.copy(), edges_end_walk.copy()], ignore_index=True),
    crs=G_walk_proj.graph["crs"],
)
wait_edges_preview_gdf = make_edges_gdf(edges_wait.copy(), crs=G_walk_proj.graph["crs"])
alight_edges_preview_gdf = make_edges_gdf(edges_alight.copy(), crs=G_walk_proj.graph["crs"])
direct_edges_preview_gdf = make_edges_gdf(edges_direct.copy(), crs=G_walk_proj.graph["crs"])
transfer_edges_preview_gdf = make_edges_gdf(edges_transfer.copy(), crs=G_walk_proj.graph["crs"])

create_interactive_map(
    boundary_gdf=study_area_gdf,
    output_html_path=PREVIEW_STITCHED_HTML,
    title="Whole Iligan - Stitched Layered Travel Graph Preview",
    walk_nodes_gdf=walk_nodes_preview_gdf,
    ride_nodes_gdf=ride_nodes_preview_gdf,
    walk_edges_gdf=walk_edges_preview_gdf,
    ride_edges_gdf=ride_edges_preview_gdf,
    wait_edges_gdf=wait_edges_preview_gdf,
    alight_edges_gdf=alight_edges_preview_gdf,
    direct_edges_gdf=direct_edges_preview_gdf,
    transfer_edges_gdf=transfer_edges_preview_gdf,
)
show_html_preview(PREVIEW_STITCHED_HTML)


## 22. Validation summary

In [119]:

validation_summary = pd.DataFrame(
    {
        "component": [
            "nodes_uncategorized",
            "nodes_walk",
            "nodes_ride",
            "edges_start_walk",
            "edges_end_walk",
            "edges_ride",
            "edges_wait",
            "edges_alight",
            "edges_direct",
            "edges_transfer",
            "travel_graph_nodes",
            "travel_graph_edges",
        ],
        "count": [
            len(nodes_uncategorized),
            len(nodes_walk),
            len(nodes_ride),
            len(edges_start_walk),
            len(edges_end_walk),
            len(edges_ride),
            len(edges_wait),
            len(edges_alight),
            len(edges_direct),
            len(edges_transfer),
            len(travel_graph_nodes),
            len(travel_graph_edges),
        ],
    }
)
validation_summary


,component,count
0,nodes_uncategorized,4758
1,nodes_walk,4689
2,nodes_ride,3477
3,edges_start_walk,11820
4,edges_end_walk,11820
5,edges_ride,8834
6,edges_wait,3448
7,edges_alight,3448
8,edges_direct,4689
9,edges_transfer,3448
